# Part 13 — Post-hoc robustness checks requested by reviewers (Papers 1 and 2)
Nothing here is a pre-registered test: every check reuses data already opened in Parts 4–12, and the papers label each result **post hoc**. There is nothing to e-mail first and no cell that must be run only once.

**Run U0, U1, U2, then R1–R6, R8, R7 in the order shown.** Most cells use files already in your Drive from earlier parts. R5 downloads 24 Bybit order-book files (~10 min), R7 downloads 192 Binance 1-second files (~30–60 min). R7 zips everything and downloads `p13_results.zip`; send it to Claude. If one cell errors, skip it and continue; tell Claude which.

| Cell | Question |
|---|---|
| R1 | Does the certified guard help when used as the theorem's one-sided throttle instead of inverse sizing? |
| R2 | Equilibrium test with the unit-free gain multiple and physical-unit S by liquidity tercile |
| R3 | Bybit vs Binance minute bars: bid–ask bounce diagnostics |
| R4 | Simultaneity: instrumented (lagged-flow) estimate of the amplification margin |
| R5 | Depth depletion on a fixed price grid (Bybit full order books) |
| R6 | CFTC: is the negative response of leveraged funds a basis-trade signature? |
| R8 | VPIN head-to-head: does C3 survive VPIN? |
| R7 | Paper 1: Cochrane–Orcutt feedback estimates on real spot data (zips the results) |

In [ ]:
# ============================================================================
# CELL U0 — SETUP (self-contained; run first; same code as Parts 9 and 12)
# Part 13: POST-HOC ROBUSTNESS CHECKS requested by reviewers. Nothing here is a pre-registered test: every check uses
# data already opened in Parts 4-12 and is labelled post hoc in the papers.
# ============================================================================
import os, sys, json, time, math, glob, re, io, zipfile, hashlib, subprocess, warnings, zlib
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed
import xml.etree.ElementTree as ET

IN_COLAB = "google.colab" in sys.modules
try:
    import pyarrow  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow"], check=True)
import numpy as np
import pandas as pd
from scipy import stats, optimize
warnings.filterwarnings("ignore", category=RuntimeWarning)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
elif IN_COLAB:
    BASE = "/content/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
XDATA = os.path.join(BASE, "data_xsec")
for _d in (RESULTS, PREREG, XDATA):
    os.makedirs(_d, exist_ok=True)

SEED = 20261029
def rng(offset=0):
    return np.random.default_rng(SEED + offset)

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    assert kind in {"theorem-check", "simulation", "real-data", "audit"}, kind
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "seed": SEED}
    with open(os.path.join(RESULTS, f"{test_id}.json"), "w") as f:
        json.dump(out, f, indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    if note:
        print("        note:", note)
    return out

def boot_ci(v, R, B=2000, stat=np.mean):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 3:
        return [np.nan, np.nan]
    bs = [stat(v[R.integers(0, len(v), len(v))]) for _ in range(B)]
    return [float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))]

# ---- Sample design --------------------------------------------------------------------------
FRESH_DAYS_OF_MONTH = (6, 20)   # never used in Parts 1-8
FRESH_START, FRESH_END = "2025-01-01", "2026-09-30"
N_SYMBOLS = 30                  # symbols in the cross-section, spread across liquidity (BTC and ETH always in)
PANEL_DAYS_OF_MONTH = (10, 24)  # same panel design as Part 1
PANEL_START, PANEL_END = "2023-01-01", "2026-09-30"
TRAIN_END, TEST_START = "2024-12-31", "2025-01-01"
LIQ_RANK_DATE = "2024-06-10"    # liquidity ranking day for choosing symbols (training period)
THREADS = 12
ROOT = "https://data.binance.vision/data/futures/um/daily"
print("BASE =", BASE)

def _http(url, method="GET", timeout=60, tries=3):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method=method, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, (r.read() if method == "GET" else b"")
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(1.5 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")

def ds_url(ds, sym, date):
    if ds == "klines":
        return f"{ROOT}/klines/{sym}/1m/{sym}-1m-{date}.zip"
    return f"{ROOT}/{ds}/{sym}/{sym}-{ds}-{date}.zip"

def exists(ds, sym, date):
    return _http(ds_url(ds, sym, date), "HEAD", timeout=20)[0] == 200

DL_LOG = []
def fetch(ds, sym, date):
    """Zip bytes, checked against Binance's published .CHECKSUM when available; None if missing."""
    url = ds_url(ds, sym, date)
    code, raw = _http(url, timeout=180)
    if code != 200:
        return None
    sha = hashlib.sha256(raw).hexdigest()
    c2, chk = _http(url + ".CHECKSUM", timeout=30)
    verified = None
    if c2 == 200 and chk:
        if chk.decode().split()[0] != sha:
            raise RuntimeError(f"SHA-256 mismatch {url}")
        verified = True
    DL_LOG.append({"ds": ds, "symbol": sym, "date": date, "sha256": sha, "checksum_verified": verified})
    return raw

def read_zip_csv(raw, default_cols=None):
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".csv")][0]
        txt = z.read(name).decode("utf-8", errors="ignore")
    first = txt.split("\n", 1)[0]
    has_header = bool(re.search(r"[A-Za-z_]", first.split(",")[0]))
    df = pd.read_csv(io.StringIO(txt), header=0 if has_header else None)
    if not has_header and default_cols is not None:
        df.columns = default_cols[:df.shape[1]]
    df.columns = [str(c).strip().lower() for c in df.columns]
    return df

def to_utc_seconds(col):
    s = pd.Series(col)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("int64").to_numpy()
        return np.where(v > 1e14, v // 1_000_000, np.where(v > 1e11, v // 1000, v))
    ts = pd.to_datetime(s, utc=True)
    return ((ts - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta("1s")).astype("int64").to_numpy()

def panel_dates(start=PANEL_START, end=PANEL_END):
    ds = pd.date_range(start, end, freq="D")
    return [d.strftime("%Y-%m-%d") for d in ds if d.day in PANEL_DAYS_OF_MONTH]

def fresh_dates(start=FRESH_START, end=FRESH_END):
    ds = pd.date_range(start, end, freq="D")
    return [d.strftime("%Y-%m-%d") for d in ds if d.day in FRESH_DAYS_OF_MONTH]
def train_dates():
    return [d for d in panel_dates() if d <= TRAIN_END]

In [ ]:
# ============================================================================
# CELL U1 — FUNCTIONS (same code as Part 6, plus the Part 9 volatility benchmarks at the end)
# ============================================================================
FALLBACK = ("BTCUSDT ETHUSDT BNBUSDT XRPUSDT ADAUSDT DOGEUSDT SOLUSDT DOTUSDT LTCUSDT LINKUSDT BCHUSDT TRXUSDT "
            "AVAXUSDT ATOMUSDT ETCUSDT FILUSDT NEARUSDT AAVEUSDT UNIUSDT SANDUSDT MANAUSDT AXSUSDT XLMUSDT ALGOUSDT "
            "VETUSDT EOSUSDT XTZUSDT THETAUSDT ICPUSDT EGLDUSDT CHZUSDT ENJUSDT ZILUSDT KAVAUSDT RUNEUSDT GALAUSDT "
            "CRVUSDT DYDXUSDT 1INCHUSDT COMPUSDT SNXUSDT ZECUSDT DASHUSDT NEOUSDT IOTAUSDT ONTUSDT QTUMUSDT BATUSDT "
            "ZRXUSDT KNCUSDT").split()
def list_bucket_symbols(ds="bookDepth"):
    url0 = ("https://s3-ap-northeast-1.amazonaws.com/data.binance.vision?delimiter=/"
            f"&prefix=data/futures/um/daily/{ds}/")
    syms, marker = [], ""
    for _ in range(20):
        code, raw = _http(url0 + (f"&marker={marker}" if marker else ""), timeout=60)
        if code != 200 or not raw:
            return []
        root = ET.fromstring(raw)
        ns = {"s": root.tag.split("}")[0].strip("{")} if root.tag.startswith("{") else {}
        pref = [e.text for e in (root.findall(".//s:CommonPrefixes/s:Prefix", ns) if ns else root.findall(".//CommonPrefixes/Prefix"))]
        syms += [p.rstrip("/").split("/")[-1] for p in pref]
        trunc = (root.find("s:IsTruncated", ns) if ns else root.find("IsTruncated"))
        if trunc is None or trunc.text != "true" or not pref:
            break
        marker = pref[-1]
    return sorted(set(syms))
EARLY, LATE = "2023-03-10", "2026-06-10"
def span_ok(s):
    return exists("bookDepth", s, EARLY) and exists("bookDepth", s, LATE)
KL_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume",
           "count", "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
def day_quote_volume(s):
    raw = fetch("klines", s, LIQ_RANK_DATE)
    return np.nan if raw is None else float(read_zip_csv(raw, KL_COLS)["quote_volume"].astype(float).sum())
NEED_BANDS = [-5, -2, -1, 1, 2, 5]
def nice_tick(x):
    """Snap a positive increment to the nearest 1, 2 or 5 x 10^k (exchange tick grids)."""
    if not np.isfinite(x) or x <= 0:
        return np.nan
    e = math.floor(math.log10(x)); m = x / 10 ** e
    c = min([1, 2, 5, 10], key=lambda v: abs(math.log(m / v)))
    return c * 10 ** e
def infer_tick(k):
    v = np.unique(np.round(k[["open", "high", "low", "close"]].astype(float).to_numpy().ravel(), 10))
    dv = np.diff(v); dv = dv[dv > 0]
    return nice_tick(np.quantile(dv, 0.02)) if len(dv) > 5 else np.nan
def validated_snapshots(sym, date, close_by_minute, t0):
    """Validated bookDepth snapshots: DataFrame indexed by second with bid/ask notional at 1,2,5% ($M)."""
    raw = fetch("bookDepth", sym, date)
    if raw is None:
        return None, {"snapshots": 0}
    b = read_zip_csv(raw)
    b["sec"] = to_utc_seconds(b["timestamp"])
    b["percentage"] = pd.to_numeric(b["percentage"], errors="coerce").round().astype("Int64")
    piv_n = b.pivot_table(index="sec", columns="percentage", values="notional", aggfunc="last")
    piv_d = b.pivot_table(index="sec", columns="percentage", values="depth", aggfunc="last")
    n0 = len(piv_n)
    ok = piv_n.reindex(columns=NEED_BANDS).notna().all(axis=1) & piv_d.reindex(columns=NEED_BANDS).notna().all(axis=1)
    piv_n, piv_d = piv_n[ok], piv_d[ok]
    mono = (piv_d[-2] >= piv_d[-1]) & (piv_d[-5] >= piv_d[-2]) & (piv_d[2] >= piv_d[1]) & (piv_d[5] >= piv_d[2])
    minute = ((piv_n.index.to_numpy() - t0) // 60).clip(0, 1439)
    ref = close_by_minute[minute]
    imp_b, imp_a = piv_n[-1] / piv_d[-1], piv_n[1] / piv_d[1]
    price_ok = (np.abs(imp_b / ref - 1) < 0.03) & (np.abs(imp_a / ref - 1) < 0.03)
    sig = piv_n[NEED_BANDS].round(2).astype(str).agg("|".join, axis=1)
    runlen = sig.groupby((sig != sig.shift()).cumsum()).transform("size")
    not_stale = runlen <= 10
    good = mono & price_ok & not_stale
    g = piv_n[good]
    snap = pd.DataFrame({"bid1": g[-1], "ask1": g[1], "bid2": g[-2], "ask2": g[2], "bid5": g[-5], "ask5": g[5]}) / 1e6
    st = {"snapshots": n0, "drop_V1": int((~ok).sum()), "drop_V2": int((~mono).sum()),
          "drop_V3": int((~price_ok).sum()), "drop_V4": int((~not_stale).sum())}
    return snap, st
def build_day(sym, date):
    t0 = int(pd.Timestamp(f"{date} 00:00:00", tz="UTC").timestamp())
    idx = np.arange(1440)
    raw = fetch("klines", sym, date)
    if raw is None:
        return None, {"symbol": sym, "date": date, "reason": "no klines"}
    k = read_zip_csv(raw, KL_COLS)
    tick = infer_tick(k)
    k = k.assign(m=(to_utc_seconds(k["open_time"]) - t0) // 60).drop_duplicates("m").set_index("m").reindex(idx)
    out = pd.DataFrame(index=idx)
    out["close"] = k["close"].astype(float).ffill().bfill()
    out["flow"] = (2 * k["taker_buy_quote_volume"].astype(float) - k["quote_volume"].astype(float)).fillna(0) / 1e6
    out["vol"] = k["quote_volume"].astype(float).fillna(0) / 1e6
    st = {"symbol": sym, "date": date, "tick": tick}
    snap, s2 = validated_snapshots(sym, date, out["close"].to_numpy(), t0)
    st.update(s2)
    for c in ["bid1", "ask1", "bid2", "ask2", "bid5", "ask5"]:
        out[c] = np.nan
    if snap is not None and len(snap):
        mm = ((snap.index.to_numpy() - t0) // 60).clip(0, 1439)
        sm = snap.assign(m=mm).groupby("m").last().reindex(idx).ffill(limit=2)
        for c in snap.columns:
            out[c] = sm[c].to_numpy()
    st["minute_coverage"] = float(out["bid1"].notna().mean())
    st["depth_day_ok"] = st["minute_coverage"] >= 0.80
    for c in ["oi", "top_pos_ratio", "top_acc_ratio", "acc_ratio", "taker_ratio"]:
        out[c] = np.nan
    raw = fetch("metrics", sym, date)
    if raw is not None:
        mt = read_zip_csv(raw)
        tcol = "create_time" if "create_time" in mt.columns else mt.columns[0]
        mm = ((to_utc_seconds(mt[tcol]) - t0) // 60).clip(0, 1439)
        for c, src, scale in [("oi", "sum_open_interest_value", 1e6),
                              ("top_pos_ratio", "sum_toptrader_long_short_ratio", 1.0),
                              ("top_acc_ratio", "count_toptrader_long_short_ratio", 1.0),
                              ("acc_ratio", "count_long_short_ratio", 1.0),
                              ("taker_ratio", "sum_taker_long_short_vol_ratio", 1.0)]:
            if src in mt.columns:
                s = pd.Series(pd.to_numeric(mt[src], errors="coerce").to_numpy() / scale, index=mm)
                out[c] = s.groupby(level=0).last().reindex(idx).to_numpy()   # 5-min values, not filled
    out["symbol"], out["date"] = sym, date
    return out, st
MIN_DIR = os.path.join(XDATA, "minute")
def get_day(sym, date):
    d_ = os.path.join(MIN_DIR, sym); os.makedirs(d_, exist_ok=True)
    pq, js = os.path.join(d_, f"{date}.parquet"), os.path.join(d_, f"{date}.json")
    if os.path.exists(js):
        st = json.load(open(js))
        return (pd.read_parquet(pq) if os.path.exists(pq) else None), st
    df, st = build_day(sym, date)
    if df is not None:
        df.to_parquet(pq)
    json.dump(_clean(st), open(js, "w"))
    return df, st
def prep(df, flow_scale):
    p = np.log(df["close"].to_numpy()) * 1e4
    u = df["flow"].to_numpy() / flow_scale
    tot = (df["bid1"] + df["ask1"]).to_numpy()
    d = ((df["bid1"] - df["ask1"]) / (df["bid1"] + df["ask1"])).to_numpy()
    x = p - pd.Series(p).ewm(span=60, adjust=False).mean().to_numpy()
    return p, u, d, x, tot
def ols(y, X):
    ok = np.all(np.isfinite(X), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 20:
        return None
    return np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), X[ok]]), y[ok], rcond=None)[0][1:]
def impulse_t6(a, phi, kappa, r, eps, eta, q, L=120):
    x = np.zeros(L); dd = np.zeros(L); f = np.zeros(L)
    for t in range(L):
        if t > 0:
            f[t] = (1 - eta) * f[t - 1] + eta * q * x[t - 1]
        tot = (1.0 if t == 0 else 0.0) + f[t]
        x[t] = (a * x[t - 1] + phi * dd[t - 1] if t else 0) + kappa * tot
        dd[t] = ((1 - r) * dd[t - 1] if t else 0) + eps * tot
        if abs(x[t]) > 1e9:
            return np.full(L, np.nan)
    return x
def fit_t6(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    bp = ols(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]))
    bd = ols(d[1:], np.column_stack([d[:-1], u[1:]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]
    r, eps, eta = 1 - bd[0], bd[1], 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    h = impulse_t6(a, phi, kappa, r, eps, eta, q)
    ok = np.all(np.isfinite(h)) and abs(h[0]) > 1e-12
    return {"kappa": kappa, "phi": phi, "a": a, "r": r, "eps": eps, "eta": eta, "q": q,
            "C2": phi * eps - (1 - a) * kappa, "C3": phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa,
            "peak_ratio": float(np.max(h) / h[0]) if ok else np.nan, "peak_minute": int(np.argmax(h)) if ok else -1}
def loop_gain(p, u, span=30):
    x = p - pd.Series(p).ewm(span=span, adjust=False).mean().to_numpy()
    f = np.roll(u, -1)[:-1]; x = x[:-1]
    Z = np.column_stack([x, f])
    Y, X = Z[1:], np.column_stack([np.ones(len(Z) - 1), Z[:-1]])
    ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(X), 1)
    if ok.sum() < 100:
        return np.nan
    J = np.linalg.lstsq(X[ok], Y[ok], rcond=None)[0][1:].T
    return float(J[0, 1] * J[1, 0] / (1 - J[1, 1])) if abs(1 - J[1, 1]) > 1e-9 else np.nan
def auc_(y, s):
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    rk = stats.rankdata(np.concatenate([pos, neg]))
    return (rk[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))
def cw_test(y, fb, ff, lags=6):
    adj = (y - fb) ** 2 - ((y - ff) ** 2 - (fb - ff) ** 2)
    m = adj.mean(); e = adj - m; n = len(adj)
    lrv = e @ e / n + 2 * sum((1 - l / (lags + 1)) * (e[l:] @ e[:-l]) / n for l in range(1, lags + 1))
    t_ = m / np.sqrt(lrv / n)
    return float(t_), float(1 - stats.norm.cdf(t_))
def design(D, cols, syms):
    X = [np.ones(len(D))] + [D[c].to_numpy() for c in cols] + [(D["symbol"] == s).to_numpy(float) for s in syms[1:]]
    return np.column_stack(X)
def evaluate(sub, outcome, cand):
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    lo, hi = A0[cand].quantile([0.01, 0.99]); A0["cand_w"] = A0[cand].clip(lo, hi); B0["cand_w"] = B0[cand].clip(lo, hi)
    need = CTRL + ["cand_w", outcome]
    A = A0.dropna(subset=need); B = B0.dropna(subset=need).sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(B) < 200 or A[outcome].std() < 1e-8 or A["cand_w"].std() < 1e-8:
        return {"skipped": "too few blocks or no variation in outcome/candidate", "n_train": len(A), "n_test": len(B),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    B = B[B["symbol"].isin(set(A["symbol"]))]
    syms = sorted(A["symbol"].unique()); y = B[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, CTRL, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, CTRL + ["cand_w"], syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(B, CTRL, syms) @ bb; ff = design(B, CTRL + ["cand_w"], syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(88); dd = B["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(2000):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    return {"n_train": len(A), "n_test": len(B), "n_symbols": len(syms),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, .025)), float(np.nanquantile(dR2, .975))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, .025)), float(np.nanquantile(dA, .975))],
            "clark_west_t": t_, "clark_west_p": p_, "coef_candidate_train": float(bf[len(CTRL) + 1])}

# ---------------------------------------------------------------------------------------------
# Part 6 additions
# ---------------------------------------------------------------------------------------------
H_GUARD = 60          # guard horizon (minutes) for the summed and peak impulse response
CTRL = ["rv_60", "rv_360", "abs_ret_60", "flow_imb_60", "log_vol_60", "log_depth1", "imbalance", "oi_chg_60"]
IMPACT = ["kappa_hat", "kappa_sq", "kyle_lam", "amihud_60"]
TAGGED = ["n_top_pos", "d_top_pos_60", "n_all", "d_all_60"]

def _last_valid(v, lo, hi):
    w = v[lo:hi]; w = w[np.isfinite(w)]
    return w[-1] if len(w) else np.nan

def day_blocks6(s, d, flow_scale):
    """One row per hour 06-23 UTC. Features use minutes [t-360, t); outcomes use (t, t+60]."""
    df = PANEL[(s, d)]
    p, u, dd_, x, tot = prep(df, flow_scale)
    oi = df["oi"].ffill().to_numpy(); vol = df["vol"].to_numpy(); dp = np.diff(p, prepend=np.nan)
    ntag = {c: ((df[col] - 1) / (df[col] + 1)).to_numpy() for c, col in [("top_pos", "top_pos_ratio"), ("all", "acc_ratio")]}
    rows = []
    for hr in range(6, 24):
        t = hr * 60
        w, h, f = slice(t - 360, t), slice(t - 60, t), slice(t, min(t + 61, 1440))
        est = fit_t6(p[w], u[w], dd_[w], x[w])
        row = {"symbol": s, "date": d, "hour": hr}
        if est is not None:
            t1, t2, t3 = est["phi"] * est["eps"], est["kappa"] ** 2 * est["eta"] * est["q"], (1 - est["a"]) * est["kappa"]
            imp = impulse_t6(est["a"], est["phi"], est["kappa"], est["r"], est["eps"], est["eta"], est["q"], L=H_GUARD)
            ok = np.all(np.isfinite(imp)) and abs(imp[0]) > 1e-12
            S60 = float(np.sum(np.abs(imp))) if ok else np.nan
            P60 = float(np.max(np.abs(imp))) if ok else np.nan
            row.update({"C3_hat": est["C3"], "term_depth": t1, "term_momentum": t2, "term_reversion": t3,
                        "kappa_hat": est["kappa"], "a_hat": est["a"], "S60": S60, "P60": P60,
                        "gain_multiple": S60 / abs(imp[0]) if ok else np.nan,
                        "log_halflife": float(np.log(1.0 / (1 - est["a"]))) if 0 < 1 - est["a"] < 1 else np.nan})
        # depth-free version of C3 (prices and flow only), for venues without depth data (Part 7)
        bp0 = ols(dp[w][1:], np.column_stack([u[w][1:], x[w][:-1]]))
        bm0 = ols(u[w][1:], np.column_stack([u[w][:-1], x[w][:-1]]))
        if bp0 is not None and bm0 is not None and abs(1 - bm0[0]) > 1e-6:
            k0, a0, e0 = bp0[0], 1 + bp0[1], 1 - bm0[0]
            row["C3_flow"] = k0 ** 2 * e0 * (bm0[1] / e0) - (1 - a0) * k0
        uu, yy = u[w][1:], dp[w][1:]; okk = np.isfinite(uu) & np.isfinite(yy)
        kl = float(np.polyfit(uu[okk], yy[okk], 1)[0]) if okk.sum() > 60 and np.std(uu[okk]) > 0 else np.nan
        v60 = np.nansum(vol[h]); au = np.abs(u[w]); au = au[np.isfinite(au)]
        row.update({
            "rv_60": np.nanstd(np.diff(p[h])), "rv_360": np.nanstd(np.diff(p[w])), "abs_ret_60": abs(p[t - 1] - p[t - 60]),
            "flow_imb_60": np.nansum(u[h]) / (np.nansum(np.abs(u[h])) + 1e-12), "log_vol_60": np.log1p(v60),
            "log_depth1": np.log(tot[t - 1]) if np.isfinite(tot[t - 1]) and tot[t - 1] > 0 else np.nan,
            "imbalance": dd_[t - 1],
            "oi_chg_60": (oi[t - 1] / oi[t - 60] - 1) * 100 if np.isfinite(oi[t - 60]) and oi[t - 60] > 0 else np.nan,
            "kyle_lam": kl,
            "amihud_60": np.log(abs(p[t - 1] - p[t - 60]) / v60 + 1e-9) if v60 > 0 and np.isfinite(p[t - 60]) else np.nan,
            "U6": float(np.quantile(au, 0.95)) if len(au) > 60 else np.nan,
            "n_top_pos": _last_valid(ntag["top_pos"], t - 60, t),
            "d_top_pos_60": _last_valid(ntag["top_pos"], t - 60, t) - _last_valid(ntag["top_pos"], t - 120, t - 60),
            "n_all": _last_valid(ntag["all"], t - 60, t),
            "d_all_60": _last_valid(ntag["all"], t - 60, t) - _last_valid(ntag["all"], t - 120, t - 60),
            "MAE_60": float(np.nanmax(np.abs(p[f] - p[t - 1]))),
            "umax_next": float(np.nanmax(np.abs(u[f])))})
        rows.append(row)
    return rows

def winsor_train(D, cols):
    D = D.copy()
    for c in cols:
        lo, hi = D.loc[D["split"] == "train", c].quantile([0.01, 0.99])
        D[c] = D[c].clip(lo, hi)
    return D

def evaluate_g(sub, outcome, cands, ctrl, B=2000, level=0.95, seed=88):
    """Out-of-sample evaluation, as in Parts 4-5, for one or more candidates on top of `ctrl`.
    Candidates are winsorized at training 1%/99%; symbol fixed effects; date-block bootstrap."""
    cands = [cands] if isinstance(cands, str) else list(cands)
    A0 = sub[sub["split"] == "train"].copy(); B0 = sub[sub["split"] == "test"].copy()
    cw = []
    for c in cands:
        lo, hi = A0[c].quantile([0.01, 0.99]); A0[c + "_w"] = A0[c].clip(lo, hi); B0[c + "_w"] = B0[c].clip(lo, hi); cw.append(c + "_w")
    need = ctrl + cw + [outcome]
    A = A0.dropna(subset=need); Bt = B0.dropna(subset=need)
    Bt = Bt[Bt["symbol"].isin(set(A["symbol"]))].sort_values(["date", "symbol", "hour"])
    if len(A) < 200 or len(Bt) < 200 or A[outcome].std() < 1e-8 or any(A[c].std() < 1e-12 for c in cw):
        return {"skipped": "too few blocks or no variation", "n_train": len(A), "n_test": len(Bt),
                "clark_west_p": np.nan, "delta_r2": np.nan, "delta_r2_ci": [np.nan, np.nan]}
    syms = sorted(A["symbol"].unique()); y = Bt[outcome].to_numpy()
    bb = np.linalg.lstsq(design(A, ctrl, syms), A[outcome].to_numpy(), rcond=None)[0]
    bf = np.linalg.lstsq(design(A, ctrl + cw, syms), A[outcome].to_numpy(), rcond=None)[0]
    fb = design(Bt, ctrl, syms) @ bb; ff = design(Bt, ctrl + cw, syms) @ bf
    ybar = A[outcome].mean(); sst = np.sum((y - ybar) ** 2)
    t_, p_ = cw_test(y, fb, ff)
    tail = (y > A[outcome].quantile(0.9)).astype(int)
    Rb = rng(seed); dd = Bt["date"].to_numpy(); ud = np.unique(dd)
    pos = {d: np.where(dd == d)[0] for d in ud}; dR2, dA = [], []
    for _ in range(B):
        idx = np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])
        s_ = np.sum((y[idx] - ybar) ** 2)
        dR2.append((np.sum((y[idx] - fb[idx]) ** 2) - np.sum((y[idx] - ff[idx]) ** 2)) / s_)
        dA.append(auc_(tail[idx], ff[idx]) - auc_(tail[idx], fb[idx]))
    qa = (1 - level) / 2
    return {"n_train": len(A), "n_test": len(Bt), "n_symbols": len(syms), "n_test_dates": len(ud),
            "r2_base": 1 - np.sum((y - fb) ** 2) / sst, "r2_full": 1 - np.sum((y - ff) ** 2) / sst,
            "delta_r2": (np.sum((y - fb) ** 2) - np.sum((y - ff) ** 2)) / sst,
            "delta_r2_ci": [float(np.nanquantile(dR2, qa)), float(np.nanquantile(dR2, 1 - qa))],
            "auc_base": auc_(tail, fb), "auc_full": auc_(tail, ff),
            "delta_auc_ci": [float(np.nanquantile(dA, qa)), float(np.nanquantile(dA, 1 - qa))],
            "clark_west_t": t_, "clark_west_p": p_, "ci_level": level,
            "coef_candidates_train": {c: float(bf[len(ctrl) + 1 + i]) for i, c in enumerate(cands)}}

# ---- Guard backtest (hypothesis B) ----------------------------------------------------------
# A dealer sizes its position each hour as q = c_s / R_t, where R_t is a rule's risk measure from
# the past 6 hours; c_s makes the TRAINING mean of q equal to 1 for every symbol and rule. The
# hour's loss is L = q * MAE_60 (worst adverse excursion, either direction). Tail-loss ratio:
#   TLR = ES95(L) / mean(q)   on the test sample: tail loss per unit of average position (lower is better).
GUARD_RULES = {"guard_summed": lambda D: D["U6"] * D["S60"],        # Paper 2 guard: shock envelope x summed |response|
               "guard_peak":   lambda D: D["U6"] * D["P60"],        # peak-response guard
               "vol":          lambda D: D["rv_360"],               # volatility-scaled limit
               "static":       lambda D: pd.Series(1.0, index=D.index)}

THETA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]
def choose_theta(D, gm_col="gain_multiple"):
    """Amplification-adjusted volatility limit R = rv_360 * GM^theta, theta chosen on TRAINING data only
    (minimum training TLR). theta = 0 is the plain volatility limit."""
    res = {}
    for th in THETA_GRID:
        G, _ = guard_frame(D, {"amp_vol": (lambda X, th=th: X["rv_360"] * X[gm_col] ** th)})
        Tr = G[G["split"] == "train"]
        res[th] = es95(Tr["L_amp_vol"]) / Tr["q_amp_vol"].mean()
    return min(res, key=res.get), res

def es95(v):
    v = np.asarray(v, float); q = np.quantile(v, 0.95); return float(v[v >= q].mean())

def guard_frame(D, extra_rules=None):
    rules = dict(GUARD_RULES); rules.update(extra_rules or {})
    G = D[["symbol", "date", "hour", "split", "MAE_60"]].copy()
    for r, fn in rules.items():
        G["R_" + r] = fn(D).astype(float).where(lambda v: v > 0)
    G = G.dropna(subset=["MAE_60"] + ["R_" + r for r in rules])
    for r in rules:                                          # winsorize risk at pooled training 1%/99%
        lo, hi = G.loc[G["split"] == "train", "R_" + r].quantile([0.01, 0.99])
        G["R_" + r] = G["R_" + r].clip(lo, hi)
        inv = 1.0 / G["R_" + r]
        c = inv[G["split"] == "train"].groupby(G["symbol"]).mean()
        G["q_" + r] = inv / G["symbol"].map(c)
        G["L_" + r] = G["q_" + r] * G["MAE_60"]
    G = G[G["symbol"].isin(c.index)]
    thr = G[G["split"] == "train"].groupby("symbol")["L_static"].quantile(0.95)
    for r in rules:
        G["breach_" + r] = (G["L_" + r] > G["symbol"].map(thr)).astype(float)
    return G, list(rules)

def guard_compare(G, rules, ref="vol", B=2000, level=0.95, seed=77):
    T = G[G["split"] == "test"].sort_values(["date", "symbol", "hour"])
    tlr = lambda X, r: es95(X["L_" + r]) / X["q_" + r].mean()
    point = {r: {"TLR": tlr(T, r), "mean_q": float(T["q_" + r].mean()), "breach_rate": float(T["breach_" + r].mean()),
                 "ES95_loss": es95(T["L_" + r])} for r in rules}
    Rb = rng(seed); dd = T["date"].to_numpy(); ud = np.unique(dd); pos = {d: np.where(dd == d)[0] for d in ud}
    bs = {r: [] for r in rules if r != ref}
    for _ in range(B):
        X = T.iloc[np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])]
        tr_ = tlr(X, ref)
        for r in bs:
            bs[r].append(np.log(tlr(X, r)) - np.log(tr_))
    qa = (1 - level) / 2
    comp = {r: {"log_TLR_minus_ref": float(np.log(point[r]["TLR"]) - np.log(point[ref]["TLR"])),
                "ci": [float(np.quantile(v, qa)), float(np.quantile(v, 1 - qa))],
                "share_boot_below_0": float(np.mean(np.array(v) < 0))} for r, v in bs.items()}
    return {"n_test_hours": len(T), "n_test_dates": len(ud), "ref": ref, "level": level, "point": point, "vs_ref": comp}

# ---- Paper 1 identification (hypothesis C) ---------------------------------------------------
CLASSES = {"top_position": "top_pos_ratio", "top_account": "top_acc_ratio", "all_accounts": "acc_ratio"}
def five_min(df, s):
    m = df.index[df["top_pos_ratio"].notna() | df["acc_ratio"].notna()].to_numpy()
    m = m[m >= 5]
    if len(m) < 100:
        return None
    p = np.log(df["close"].to_numpy()) * 1e4; fl = df["flow"].to_numpy() / FLOW_SCALE[s]
    cf = np.concatenate([[0.0], np.cumsum(fl)])
    out = pd.DataFrame({"m": m, "ret": p[m] - p[m - 5], "flow": cf[m + 1] - cf[m - 4]})
    for c, col in CLASSES.items():
        R = df[col].to_numpy()[m]
        out[c] = (R - 1) / (R + 1)
    return out
def tagged_fit(days5, c):
    Y, X = [], []
    for g in days5:
        n = g[c].to_numpy(); r = g["ret"].to_numpy()
        y, x1, x2 = n[1:], n[:-1], r[1:]
        ok = np.isfinite(y) & np.isfinite(x1) & np.isfinite(x2)
        if ok.sum() < 30:
            continue
        Y.append(y[ok] - y[ok].mean()); X.append(np.column_stack([x1[ok] - x1[ok].mean(), x2[ok] - x2[ok].mean()]))
    if not Y:
        return np.nan, np.nan
    b = np.linalg.lstsq(np.vstack(X), np.concatenate(Y), rcond=None)[0]
    return float(b[0]), float(b[1])
def kernel_fit(days5, L=12):
    """Price-only model: 5-min flow on 12 lagged 5-min returns; kernel k_m fitted by two exponentials.
    Also returns the merged moments the theory says prices DO identify: M0 = sum k_m, mean lag M1."""
    Y, X = [], []
    for g in days5:
        f, r = g["flow"].to_numpy(), g["ret"].to_numpy()
        if len(f) <= L + 10:
            continue
        Xl = np.column_stack([r[L - m:len(r) - m] for m in range(1, L + 1)]); y = f[L:]
        ok = np.all(np.isfinite(Xl), 1) & np.isfinite(y)
        Y.append(y[ok] - y[ok].mean()); X.append(Xl[ok] - Xl[ok].mean(0))
    k = np.linalg.lstsq(np.vstack(X), np.concatenate(Y), rcond=None)[0]
    m = np.arange(L)
    def resid(th):
        w1, r1, w2, r2 = th
        return w1 * r1 ** m + w2 * r2 ** m - k
    best = None
    for r1, r2 in [(0.2, 0.8), (0.1, 0.6), (0.4, 0.95), (0.05, 0.3)]:
        try:
            sol = optimize.least_squares(resid, [k[0] / 2, r1, k[0] / 2, r2],
                                         bounds=([-np.inf, 0, -np.inf, 0], [np.inf, 0.99, np.inf, 0.99]))
            if best is None or sol.cost < best.cost:
                best = sol
        except Exception:
            pass
    w1, r1, w2, r2 = best.x
    if r1 > r2:
        w1, r1, w2, r2 = w2, r2, w1, r1
    M0 = float(k.sum()); M1 = float(np.sum(np.arange(1, L + 1) * k) / M0) if abs(M0) > 1e-12 else np.nan
    return {"k": k, "r_fast": r1, "r_slow": r2, "M0": M0, "M1": M1}

def ident_symbol(s, days, B=200, seed=90):
    days5 = [g for g in (five_min(PANEL[(s, d)], s) for d in days if (s, d) in PANEL) if g is not None]
    if len(days5) < 15:
        return None
    R9 = rng(seed); n = len(days5)
    point = {c: tagged_fit(days5, c) for c in CLASSES}; kp = kernel_fit(days5)
    bt = {c: [] for c in CLASSES}; bk = []
    for _ in range(B):
        pick = [days5[i] for i in R9.integers(0, n, n)]
        for c in CLASSES:
            bt[c].append(tagged_fit(pick, c))
        try:
            kk = kernel_fit(pick); bk.append((kk["r_fast"], kk["r_slow"], kk["M0"]))
        except Exception:
            pass
    q = lambda v: [float(np.nanquantile(v, .025)), float(np.nanquantile(v, .975))]
    bk = np.array(bk)
    tag = {c: {"r": point[c][0], "r_ci": q([b[0] for b in bt[c]])} for c in CLASSES}
    w_tag = float(np.mean([tag[c]["r_ci"][1] - tag[c]["r_ci"][0] for c in CLASSES]))
    rf_ci, rs_ci, M0_ci = q(bk[:, 0]), q(bk[:, 1]), q(bk[:, 2])
    w_pr = float(np.mean([rf_ci[1] - rf_ci[0], rs_ci[1] - rs_ci[0]]))
    diff = np.array([b[0] for b in bt["top_position"]]) - np.array([b[0] for b in bt["all_accounts"]])
    return {"n_days": n, "tagged": tag, "price_only": {"r_fast": kp["r_fast"], "r_slow": kp["r_slow"], "r_fast_ci": rf_ci,
            "r_slow_ci": rs_ci, "M0": kp["M0"], "M0_ci": M0_ci, "M1": kp["M1"]},
            "width_tagged": w_tag, "width_price_only": w_pr, "width_ratio": w_pr / w_tag if w_tag > 0 else np.nan,
            "M0_identified": bool(M0_ci[0] > 0 or M0_ci[1] < 0), "rates_unidentified": bool(w_pr > 0.25),
            "top_minus_all_rate_ci": q(diff)}


# ---------------------------------------------------------------------------------------------
# Part 9 additions: volatility-forecasting benchmarks (all from data before t, same day)
# ---------------------------------------------------------------------------------------------
HOURS = list(range(7, 24))                                  # hour 6 is the omitted category
VOLBENCH = ["rv_15", "rv_day", "ewma_vol", "mae_lag60"] + [f"hr_{h}" for h in HOURS]
def day_blocks9(s, d, flow_scale):
    rows = day_blocks6(s, d, flow_scale)
    df = PANEL[(s, d)]; p = np.log(df["close"].to_numpy()) * 1e4; dp = np.diff(p, prepend=np.nan)
    ew = pd.Series(np.nan_to_num(dp) ** 2).ewm(alpha=0.06, adjust=False).mean().to_numpy()   # RiskMetrics lambda = 0.94
    for row in rows:
        t = row["hour"] * 60
        row["rv_15"] = np.nanstd(dp[t - 15:t]); row["rv_day"] = np.nanstd(dp[1:t])
        row["ewma_vol"] = float(np.sqrt(ew[t - 1]))
        row["mae_lag60"] = float(np.nanmax(np.abs(p[t - 60:t] - p[t - 61])))
        for h in HOURS:
            row[f"hr_{h}"] = float(row["hour"] == h)
    return rows

In [ ]:
# ============================================================================
# CELL U2 — SYMBOLS (fixed from Parts 4-5; nothing new is chosen)
#   thin: the 10 Part 4 thin-tercile symbols + the 10 Part 5 thin symbols (20)
#   mid:  the 10 Part 4 mid-tercile symbols (negative control: theory and Parts 4-5 predict no effect)
# ============================================================================
X3 = json.load(open(os.path.join(RESULTS, "X3_T6_cross_section.json")))["metrics"]["by_symbol"]
Y3 = json.load(open(os.path.join(RESULTS, "Y3_groups.json")))["metrics"]["group"]
THIN = sorted([s for s, v in X3.items() if v["tercile"] == "thin"] + [s for s, g in Y3.items() if g == "thin"])
MID = sorted([s for s, v in X3.items() if v["tercile"] == "mid"])
GROUP = {**{s: "thin" for s in THIN}, **{s: "mid" for s in MID}}
SYMBOLS = THIN + MID
assert len(THIN) == 20 and len(MID) == 10, (len(THIN), len(MID))
print("thin (20):", THIN); print("mid (10):", MID)
record("U1_universe", None, "audit", {"thin": THIN, "mid": MID})

## R1

In [ ]:
# ============================================================================
# CELL R1 — THE GUARD AS A ONE-SIDED THROTTLE (post hoc; Part 12 blocks)
# Theorem 5.1's guard caps admission at min(1, B / (I U S)); it never levers up. Part 12 sized positions as c / R, which
# levers up when the estimated response is small. Here the certified guard only scales a volatility-sized position down.
# ============================================================================
QB = pd.read_parquet(os.path.join(XDATA, "blocks_p12.parquet")).dropna(subset=["MAE_60", "rv_360", "U_cert", "S60_cert"])
QB = QB[(QB["rv_360"] > 0) & (QB["U_cert"] * QB["S60_cert"] > 0)].copy()
tr = QB["split"] == "train"
for c in ["rv_360"]:
    lo, hi = QB.loc[tr, c].quantile([0.01, 0.99]); QB[c] = QB[c].clip(lo, hi)
QB["US"] = QB["U_cert"] * QB["S60_cert"]
kmed = QB[tr].groupby("symbol")["US"].median()
QB["throttle"] = np.minimum(1.0, QB["symbol"].map(kmed) / QB["US"])
def norm_q(raw):
    c = raw[tr].groupby(QB.loc[tr, "symbol"]).mean()
    return raw / QB["symbol"].map(c)
QB["q_vol"] = norm_q(1.0 / QB["rv_360"])
QB["q_vol_thr"] = norm_q((1.0 / QB["rv_360"]) * QB["throttle"])
QB["q_static_thr"] = norm_q(QB["throttle"])
QB["q_static"] = 1.0
def tlr(X, col):
    L = X[col] * X["MAE_60"]; return es95(L) / X[col].mean()
T = QB[~tr].sort_values(["date", "symbol", "hour"]); dd = T["date"].to_numpy(); ud = np.unique(dd)
pos = {d: np.where(dd == d)[0] for d in ud}; Rb = rng(131)
cols = ["q_vol", "q_vol_thr", "q_static", "q_static_thr"]
pt = {c: tlr(T, c) for c in cols}
bs = {c: [] for c in cols if c != "q_vol"}
for _ in range(2000):
    X = T.iloc[np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])]; ref = tlr(X, "q_vol")
    for c in bs:
        bs[c].append(np.log(tlr(X, c)) - np.log(ref))
r1 = {c: {"TLR": pt[c], "vs_vol": float(np.log(pt[c]) - np.log(pt["q_vol"])) if c != "q_vol" else 0.0,
          "ci": [float(np.quantile(bs[c], .025)), float(np.quantile(bs[c], .975))] if c in bs else None,
          "share_throttled": float((T["throttle"] < 1).mean())} for c in cols}
b2 = []
for _ in range(2000):
    X = T.iloc[np.concatenate([pos[d] for d in Rb.choice(ud, len(ud))])]
    b2.append(np.log(tlr(X, "q_static_thr")) - np.log(tlr(X, "q_static")))
r1["static_thr_vs_static"] = {"est": float(np.log(pt["q_static_thr"]) - np.log(pt["q_static"])),
                              "ci": [float(np.quantile(b2, .025)), float(np.quantile(b2, .975))]}
print(json.dumps(r1, indent=1, default=float))
record("R1_guard_throttle", None, "real-data", r1, note="post hoc; Part 12 test hours")

## R2

In [ ]:
# ============================================================================
# CELL R2 — EQUILIBRIUM PREDICTION ACROSS LIQUIDITY TERCILES (post hoc; Part 4 training days, cached)
# Reports the unit-free gain multiple S/|h0| = 1/(m - q kappa~) and the summed response in physical units
# (basis points per $1M of flow) by tercile, instead of flow-normalized S.
# ============================================================================
X3 = json.load(open(os.path.join(RESULTS, "X3_T6_cross_section.json")))["metrics"]["by_symbol"]
TERC = {s: v["tercile"] for s, v in X3.items()}
TR_D = [d for d in panel_dates() if d <= TRAIN_END]
jobs = [(s, d) for s in TERC for d in TR_D]
PAN2 = {}
with ThreadPoolExecutor(THREADS) as ex:
    for (s, d), (df, st) in zip(jobs, ex.map(lambda sd: get_day(*sd), jobs)):
        if df is not None and st.get("depth_day_ok"):
            PAN2[(s, d)] = df
FS2 = {}
for s in TERC:
    v = [np.nanmedian(np.abs(PAN2[(s, d)]["flow"].to_numpy())) for d in TR_D if (s, d) in PAN2]
    FS2[s] = float(np.nanmedian(v)) if v and np.nanmedian(v) > 0 else np.nan
rows = []
for (s, d), df in PAN2.items():
    if not np.isfinite(FS2[s]):
        continue
    p, u, dd_, x, tot = prep(df, FS2[s]); est = fit_t6(p, u, dd_, x)
    if est is None:
        continue
    imp = impulse_t6(est["a"], est["phi"], est["kappa"], est["r"], est["eps"], est["eta"], est["q"], L=H_GUARD)
    if not (np.all(np.isfinite(imp)) and abs(imp[0]) > 1e-12):
        continue
    S = float(np.sum(np.abs(imp)))
    rows.append({"symbol": s, "date": d, "tercile": TERC[s], "S60_norm": S, "GM": S / abs(imp[0]),
                 "S60_bp_per_musd": S / FS2[s], "depth1_musd": float(np.nanmedian(tot))})
E2 = pd.DataFrame(rows)
sym = E2.groupby(["symbol", "tercile"])[["S60_norm", "GM", "S60_bp_per_musd", "depth1_musd"]].median().reset_index()
Rb = rng(132); r2 = {"by_tercile": {}, "spearman_with_log_depth": {}}
for t, G in sym.groupby("tercile"):
    r2["by_tercile"][t] = {c: {"median": float(G[c].median()),
                              "ci": boot_ci(G[c].to_numpy(), Rb, stat=np.median)} for c in ["S60_norm", "GM", "S60_bp_per_musd"]}
for c in ["S60_norm", "GM", "S60_bp_per_musd"]:
    r2["spearman_with_log_depth"][c] = float(stats.spearmanr(np.log(sym["depth1_musd"]), sym[c])[0])
print(json.dumps(r2, indent=1))
record("R2_terciles_gain_multiple", None, "real-data", r2, note="post hoc; Part 4 training days; symbol-level medians")

## R3

In [ ]:
# ============================================================================
# CELL R3 — WHY DID BYBIT DIFFER? MINUTE-BAR DIAGNOSTICS (post hoc; cached Part 7 and Part 4 minute bars)
# Same coins, same days (10th and 24th of each month, 2023-2024): Bybit bars built from last trades (Part 7) vs Binance
# kline closes (Part 4). Bid-ask bounce shows up as negative first-order autocorrelation of one-minute returns and as
# zero-return or no-trade minutes.
# ============================================================================
BYB = os.path.join(BASE, "data_bybit", "minute")
coins = sorted(d for d in os.listdir(BYB)) if os.path.isdir(BYB) else []
r3 = {}
for s in coins:
    st = {"bybit": [], "binance": []}
    for d in [x for x in panel_dates() if x <= TRAIN_END]:
        fb = os.path.join(BYB, s, f"{d}.parquet"); fn = os.path.join(XDATA, "minute", s, f"{d}.parquet")
        if not (os.path.exists(fb) and os.path.exists(fn)):
            continue
        for venue, fp in (("bybit", fb), ("binance", fn)):
            B = pd.read_parquet(fp); p = np.log(B["close"].to_numpy(float)); r = np.diff(p); v = B["vol"].to_numpy(float)
            ok = np.isfinite(r)
            if ok.sum() < 500 or np.std(r[ok]) == 0:
                continue
            st[venue].append({"ac1": float(np.corrcoef(r[ok][1:], r[ok][:-1])[0, 1]), "zero_ret": float(np.mean(r[ok] == 0)),
                              "no_volume": float(np.mean(v <= 0)), "rv_bp": float(np.std(r[ok]) * 1e4)})
    if st["bybit"] and st["binance"]:
        r3[s] = {vn: {k: float(np.median([x[k] for x in L])) for k in ("ac1", "zero_ret", "no_volume", "rv_bp")} | {"days": len(L)}
                 for vn, L in st.items()}
summ = {vn: {k: float(np.median([r3[s][vn][k] for s in r3])) for k in ("ac1", "zero_ret", "no_volume", "rv_bp")} for vn in ("bybit", "binance")}
print(pd.DataFrame({s: {f"{vn}_{k}": r3[s][vn][k] for vn in ("bybit", "binance") for k in ("ac1", "zero_ret", "no_volume")} for s in r3}).T.round(3).to_string())
print("medians across coins:", json.dumps(summ, indent=1))
record("R3_bybit_bar_diagnostics", None, "real-data", {"by_coin": r3, "median_across_coins": summ, "coins": len(r3)},
       note="post hoc; same coins and days on both venues")

## R4

In [ ]:
# ============================================================================
# CELL R4 — SIMULTANEITY: AN INSTRUMENTED ESTIMATE OF THE AMPLIFICATION MARGIN (post hoc; Part 9 days and split)
# The baseline T6 fit regresses the price change on same-minute flow. Here kappa and the depletion slope are estimated by
# two-stage least squares with last minute's flow as the instrument for this minute's flow, and the resulting C3_iv is
# evaluated out of sample exactly as in Part 9 (same blocks, same baseline, same split).
# ============================================================================
def tsls(y, X, Z):
    ok = np.all(np.isfinite(X), 1) & np.all(np.isfinite(Z), 1) & np.isfinite(y)
    if ok.sum() < X.shape[1] + 30:
        return None, np.nan
    X1 = np.column_stack([np.ones(ok.sum()), X[ok]]); Z1 = np.column_stack([np.ones(ok.sum()), Z[ok]])
    Pz = Z1 @ np.linalg.lstsq(Z1, X1, rcond=None)[0]
    b = np.linalg.lstsq(Pz, y[ok], rcond=None)[0][1:]
    fs = np.corrcoef(X[ok][:, 0], Z[ok][:, 0])[0, 1]
    return b, fs
def fit_t6_iv(p, u, d, x):
    dp = np.diff(p, prepend=np.nan)
    ul = np.r_[np.nan, u[:-1]]
    bp, fs1 = tsls(dp[1:], np.column_stack([u[1:], d[:-1], x[:-1]]), np.column_stack([ul[1:], d[:-1], x[:-1]]))
    bd, fs2 = tsls(d[1:], np.column_stack([u[1:], d[:-1]]), np.column_stack([ul[1:], d[:-1]]))
    bm = ols(u[1:], np.column_stack([u[:-1], x[:-1]]))
    if bp is None or bd is None or bm is None:
        return None
    kappa, phi, a = bp[0], bp[1], 1 + bp[2]; eps, r = bd[0], 1 - bd[1]; eta = 1 - bm[0]
    q = bm[1] / eta if abs(eta) > 1e-6 else np.nan
    return {"C3_iv": phi * eps + kappa ** 2 * eta * q - (1 - a) * kappa, "kappa_iv": kappa, "first_stage_corr": fs1}
UB = pd.read_parquet(os.path.join(XDATA, "blocks_p9_30sym.parquet"))
UB["group"] = UB["symbol"].map(GROUP); UB["split"] = np.where(UB["date"] <= TRAIN_END, "train", "test")
UB["kappa_sq"] = UB["kappa_hat"] ** 2
UB = winsor_train(UB, IMPACT + ["rv_15", "rv_day", "ewma_vol", "mae_lag60"])     # exactly as Part 9 cell U5
TH = UB[UB["group"] == "thin"].copy()
need = sorted(set(map(tuple, TH[["symbol", "date"]].drop_duplicates().to_numpy())))
FS9 = {}
for s in THIN:
    v = []
    for d in [x for x in panel_dates() if x <= TRAIN_END]:
        df, st = get_day(s, d)
        if df is not None:
            v.append(np.nanmedian(np.abs(df["flow"].to_numpy())))
    FS9[s] = float(np.nanmedian(v)) if v and np.nanmedian(v) > 0 else 1.0
ivrows = []
for i, (s, d) in enumerate(need):
    df, st = get_day(s, d)
    if df is None:
        continue
    p, u, dd_, x, tot = prep(df, FS9[s])
    for hr in range(6, 24):
        t = hr * 60; w = slice(t - 360, t)
        e = fit_t6_iv(p[w], u[w], dd_[w], x[w])
        if e is not None:
            ivrows.append({"symbol": s, "date": d, "hour": hr, **e})
    if (i + 1) % 200 == 0:
        print(f"   {i+1}/{len(need)}")
IV = pd.DataFrame(ivrows)
TH = TH.merge(IV, on=["symbol", "date", "hour"], how="left")
BASE9 = CTRL + IMPACT + VOLBENCH
ev_iv = evaluate_g(TH, "MAE_60", "C3_iv", BASE9)
ev_ols = evaluate_g(TH, "MAE_60", "C3_hat", BASE9)
corr = float(TH[["C3_hat", "C3_iv"]].dropna().corr(method="spearman").iloc[0, 1])
sh = lambda v: {k: v.get(k) for k in ["n_test", "delta_r2", "delta_r2_ci", "clark_west_t", "clark_west_p", "coef_candidates_train"]}
print("C3 (OLS, as Part 9):", sh(ev_ols)); print("C3 (IV):", sh(ev_iv)); print("Spearman(C3_hat, C3_iv) =", round(corr, 3),
      "| median first-stage corr(u_t, u_t-1) =", round(float(IV["first_stage_corr"].median()), 3))
record("R4_iv_C3", None, "real-data", {"iv": ev_iv, "ols_reproduction": ev_ols, "spearman_ols_iv": corr,
       "median_first_stage_corr": float(IV["first_stage_corr"].median()), "median_kappa_iv_over_ols": float((TH["kappa_iv"] / TH["kappa_hat"]).median())},
       note="post hoc; Part 9 thin-book split, already opened")

## R5

In [ ]:
# ============================================================================
# CELL R5 — DEPLETION ON A FIXED PRICE GRID (post hoc; Bybit order-book files, the six Part 10 coins)
# Binance's bookDepth bands are re-centred on the current mid, so a price move also moves the band. Bybit's order-book
# files contain every level, so depth can be measured in a band fixed at the previous mid. For 30-second intervals we
# regress the change in depth imbalance on signed taker flow, with the band (a) fixed at the start-of-interval mid and
# (b) re-centred at the end-of-interval mid (Binance's construction).
# ============================================================================
import gzip
R5_COINS = ["ACH", "ALICE", "API3", "LIT", "WOO", "ZEN"]
R5_DAYS = ["2025-05-11", "2025-05-12", "2025-09-11", "2025-09-12"]
OBURL = "https://quote-saver.bycsi.com/orderbook/linear/{s}/{d}_{s}_ob{n}.data.zip"
def bybit_trades(sym, date):
    code, raw = _http(f"https://public.bybit.com/trading/{sym}/{sym}{date}.csv.gz", timeout=180)
    if code != 200 or not raw:
        return None
    tr = pd.read_csv(io.BytesIO(gzip.decompress(raw))); tr.columns = [c.strip().lower() for c in tr.columns]
    ts = pd.to_numeric(tr["timestamp"], errors="coerce").to_numpy(); ts = np.where(ts > 1e11, ts / 1000.0, ts)
    px = pd.to_numeric(tr["price"], errors="coerce").to_numpy(); sz = pd.to_numeric(tr["size"], errors="coerce").to_numpy()
    sg = np.where(tr["side"].astype(str).str.lower().str.startswith("b"), 1.0, -1.0)
    return pd.DataFrame({"ts": ts, "f": sg * px * sz})
def band_imb(bids, asks, mid, band=0.01):
    B = sum(p * q for p, q in bids.items() if p >= mid * (1 - band)); A = sum(p * q for p, q in asks.items() if p <= mid * (1 + band))
    return (B - A) / (B + A) if B + A > 0 else np.nan
def depletion_day(coin, date, step=30):
    sym = f"{coin}USDT"; raw = None
    for n in (200, 500):
        code, raw = _http(OBURL.format(s=sym, d=date, n=n), timeout=600)
        if code == 200 and raw:
            break
    T = bybit_trades(sym, date)
    if not raw or T is None:
        return None
    t0 = pd.Timestamp(date, tz="UTC").timestamp(); nint = 86400 // step
    flow = np.zeros(nint); k = ((T["ts"].to_numpy() - t0) // step).astype(int); ok = (k >= 0) & (k < nint)
    np.add.at(flow, k[ok], T["f"].to_numpy()[ok])
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        name = [n for n in z.namelist() if n.endswith(".data")][0]; lines = z.read(name).decode().splitlines()
    bids, asks = {}, {}; cur = None; rows = []; prev = None
    def snap(i):
        nonlocal prev
        if not bids or not asks:
            return
        bb, ba = max(bids), min(asks)
        if bb >= ba:
            return
        mid = 0.5 * (bb + ba)
        bl = dict(sorted(bids.items(), reverse=True)[:200]); al = dict(sorted(asks.items())[:200])
        if prev is not None and prev[0] == i - 1:
            m0 = prev[1]
            rows.append({"i": i, "d_fixed_end": band_imb(bl, al, m0), "d_fixed_start": prev[2],
                         "d_moving_end": band_imb(bl, al, mid), "d_moving_start": prev[3], "ret_bp": 1e4 * np.log(mid / m0)})
        prev = (i, mid, band_imb(bl, al, mid), band_imb(bl, al, mid))
    for line in lines:
        r = json.loads(line); i = int((r["ts"] / 1000 - t0) // step)
        if cur is not None and i != cur:
            snap(cur)
        cur = i; d = r["data"]
        if r["type"] == "snapshot":
            bids = {float(p): float(q) for p, q in d["b"]}; asks = {float(p): float(q) for p, q in d["a"]}
        else:
            for side, book in (("b", bids), ("a", asks)):
                for p, q in d[side]:
                    p = float(p); q = float(q)
                    if q == 0:
                        book.pop(p, None)
                    else:
                        book[p] = q
    D = pd.DataFrame(rows)
    D = D[(D["i"] >= 1) & (D["i"] < nint)]
    D["flow"] = flow[D["i"].to_numpy()] / max(np.median(np.abs(flow[flow != 0])), 1e-9)
    D["coin"], D["date"] = coin, date
    return D
parts = []
for c in R5_COINS:
    for d in R5_DAYS:
        try:
            X = depletion_day(c, d)
            if X is not None and len(X) > 500:
                parts.append(X); print(c, d, len(X))
        except Exception as e:
            print("skip", c, d, e)
DP = pd.concat(parts)
r5 = {}
for kind in ("fixed", "moving"):
    y = (DP[f"d_{kind}_end"] - DP[f"d_{kind}_start"]).to_numpy(); X = DP["flow"].to_numpy()
    ok = np.isfinite(y) & np.isfinite(X)
    by = []
    for (c, d), G in DP[ok].groupby(["coin", "date"]):
        yy = (G[f"d_{kind}_end"] - G[f"d_{kind}_start"]).to_numpy(); xx = G["flow"].to_numpy()
        by.append(float(np.polyfit(xx, yy, 1)[0]))
    b = np.array(by); Rb = rng(135)
    r5[kind] = {"eps30_mean_over_coin_days": float(b.mean()), "ci": boot_ci(b, Rb), "share_positive": float((b > 0).mean()), "coin_days": len(b)}
print(json.dumps(r5, indent=1))
record("R5_fixed_grid_depletion", None, "real-data", r5, note="post hoc; Bybit order books, six coins, four days each")

## R6

In [ ]:
# ============================================================================
# CELL R6 — CFTC: IS THE NEGATIVE RESPONSE OF LEVERAGED FUNDS A BASIS-TRADE SIGNATURE? (post hoc; Part 11 data)
# In a cash-futures basis trade, leveraged funds sell futures that asset managers (or cash-holders) buy, so their weekly
# position changes move in opposite directions. We report corr(dn_fast, dn_slow) and the fast-class response to returns
# with the slow class's position change held fixed.
# ============================================================================
CDIR = os.path.join(BASE, "data_cot")
PX = {"BTC": ("binance", "px_BTCUSDT.csv"), "ETH": ("binance", "px_ETHUSDT.csv"), "ES": ("yahoo", "yahoo_GSPC.csv"),
      "TY": ("yield", "yahoo_TNX.csv"), "EUR": ("yahoo", "yahoo_EURUSD_X.csv"), "JPY": ("inverse", "yahoo_JPY_X.csv"),
      "CL": ("yahoo", "yahoo_CL_F.csv"), "NG": ("yahoo", "yahoo_NG_F.csv")}
r6 = {}
for mkt, (kind, f) in PX.items():
    fc, fp = os.path.join(CDIR, f"cot_{mkt}.csv"), os.path.join(CDIR, f)
    if not (os.path.exists(fc) and os.path.exists(fp)):
        print("missing", mkt); continue
    cot = pd.read_csv(fc, parse_dates=["date"]); P = pd.read_csv(fp, parse_dates=["date"]).dropna()
    v = P.set_index("date").iloc[:, 0].astype(float).sort_index()
    lp = {"binance": np.log(v), "yahoo": np.log(v), "inverse": -np.log(v), "yield": -v}[kind]
    D = pd.DataFrame({"date": cot["date"], "lp": lp.reindex(cot["date"], method="ffill").to_numpy()})
    for c in ("fast", "slow"):
        D[f"n_{c}"] = ((cot[f"{c}_long"] - cot[f"{c}_short"]) / cot["oi"]).to_numpy()
        D[f"dn_{c}"] = D[f"n_{c}"].diff(); D[f"n_{c}_lag"] = D[f"n_{c}"].shift(1)
    D["ret"] = D["lp"].diff(); D = D[D["date"].diff().dt.days == 7].dropna()
    X0 = np.column_stack([np.ones(len(D)), D["n_fast_lag"], D["ret"]])
    X1 = np.column_stack([X0, D["dn_slow"]])
    b0 = np.linalg.lstsq(X0, D["n_fast"], rcond=None)[0]; b1 = np.linalg.lstsq(X1, D["n_fast"], rcond=None)[0]
    Rb = rng(136); T = len(D); bs = []
    for _ in range(1000):
        nb = int(np.ceil(T / 13)); st = Rb.integers(0, T - 13 + 1, nb); idx = np.concatenate([np.arange(s, s + 13) for s in st])[:T]
        bs.append(np.linalg.lstsq(X1[idx], D["n_fast"].to_numpy()[idx], rcond=None)[0][2])
    r6[mkt] = {"weeks": T, "corr_dn_fast_dn_slow": float(np.corrcoef(D["dn_fast"], D["dn_slow"])[0, 1]),
               "w_fast": float(b0[2]), "w_fast_given_dn_slow": float(b1[2]), "ci": [float(np.quantile(bs, .025)), float(np.quantile(bs, .975))],
               "coef_dn_slow": float(b1[3])}
print(pd.DataFrame(r6).T.round(3).to_string())
record("R6_cftc_basis_signature", None, "real-data", r6, note="post hoc; Part 11 data")

## R8

In [ ]:
# ============================================================================
# CELL R8 — HEAD-TO-HEAD WITH VPIN (post hoc; Part 9 days, split and baseline)
# VPIN (Easley, Lopez de Prado and O'Hara, 2012): mean absolute buy-sell imbalance over equal-volume buckets. Binance
# klines report taker buy volume, so the imbalance in each bucket is measured, not classified. For each hour we use the
# previous six hours (same window as C3), 50 buckets. We ask (a) does VPIN forecast beyond the Part 9 baseline, and
# (b) does C3 still forecast once VPIN is added to the baseline?
# ============================================================================
def vpin_window(vol, flow, nb=50):
    v = np.nan_to_num(vol); f = np.nan_to_num(flow); tot = v.sum()
    if tot <= 0:
        return np.nan
    buy = (v + f) / 2; cum = np.cumsum(v); edges = np.linspace(0, tot, nb + 1)
    k = np.clip(np.searchsorted(edges, cum, side="left") - 1, 0, nb - 1)     # bucket of each minute (minute-level granularity)
    imb = np.zeros(nb); vb = np.zeros(nb)
    np.add.at(imb, k, buy - (v - buy)); np.add.at(vb, k, v)
    return float(np.sum(np.abs(imb)) / tot)
UB8 = pd.read_parquet(os.path.join(XDATA, "blocks_p9_30sym.parquet"))
UB8["group"] = UB8["symbol"].map(GROUP); UB8["split"] = np.where(UB8["date"] <= TRAIN_END, "train", "test")
UB8["kappa_sq"] = UB8["kappa_hat"] ** 2
UB8 = winsor_train(UB8, IMPACT + ["rv_15", "rv_day", "ewma_vol", "mae_lag60"])
TH8 = UB8[UB8["group"] == "thin"].copy()
vrows = []
for (s, d) in sorted(set(map(tuple, TH8[["symbol", "date"]].drop_duplicates().to_numpy()))):
    df, st = get_day(s, d)
    if df is None:
        continue
    vol, flow = df["vol"].to_numpy(), df["flow"].to_numpy()
    for hr in range(6, 24):
        t = hr * 60
        vrows.append({"symbol": s, "date": d, "hour": hr, "vpin": vpin_window(vol[t - 360:t], flow[t - 360:t])})
TH8 = TH8.merge(pd.DataFrame(vrows), on=["symbol", "date", "hour"], how="left")
lo, hi = TH8.loc[TH8["split"] == "train", "vpin"].quantile([0.01, 0.99]); TH8["vpin_w"] = TH8["vpin"].clip(lo, hi)
BASE9 = CTRL + IMPACT + VOLBENCH
ev_vpin = evaluate_g(TH8, "MAE_60", "vpin", BASE9)
ev_c3_given_vpin = evaluate_g(TH8, "MAE_60", "C3_hat", BASE9 + ["vpin_w"])
ev_c3 = evaluate_g(TH8, "MAE_60", "C3_hat", BASE9)
sh = lambda v: {k: v.get(k) for k in ["n_test", "delta_r2", "delta_r2_ci", "clark_west_t", "clark_west_p"]}
print("VPIN beyond the Part 9 baseline:", sh(ev_vpin))
print("C3 beyond the Part 9 baseline + VPIN:", sh(ev_c3_given_vpin))
print("C3 beyond the Part 9 baseline (reproduction):", sh(ev_c3))
print("Spearman(C3_hat, VPIN) =", round(float(TH8[["C3_hat", "vpin"]].dropna().corr(method="spearman").iloc[0, 1]), 3))
record("R8_vpin", None, "real-data", {"vpin_beyond_baseline": ev_vpin, "c3_beyond_baseline_plus_vpin": ev_c3_given_vpin,
       "c3_beyond_baseline": ev_c3, "spearman_c3_vpin": float(TH8[["C3_hat", "vpin"]].dropna().corr(method="spearman").iloc[0, 1])},
       note="post hoc; Part 9 thin-book split, already opened")

## R7 (zips the results)

In [ ]:
# ============================================================================
# CELL R7 — PAPER 1: COCHRANE-ORCUTT ON REAL SPOT DATA (post hoc; a 2024 sub-panel of Part 8: 8 pairs x 24 dates)
# Order splitting makes flow-equation errors serially correlated, which biases OLS feedback estimates (Paper 1,
# Section 9.5). Here the VAR(1) feedback index Q is re-estimated after quasi-differencing with the estimated AR(1)
# coefficient of the flow-equation residuals.
# ============================================================================
SPOT = "https://data.binance.vision/data/spot/daily/klines"
PAIRS = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "ADAUSDT", "DOGEUSDT", "BNBUSDT", "XRPUSDT", "LTCUSDT"]
CO_D = [d.strftime("%Y-%m-%d") for d in pd.date_range("2024-01-01", "2024-12-31", freq="D") if d.day in (10, 24)]
BAR, EMA_W = 10, 300
KCOLS = ["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_volume", "count",
         "taker_buy_volume", "taker_buy_quote_volume", "ignore"]
def bars10_spot(sym, date):
    code, raw = _http(f"{SPOT}/{sym}/1s/{sym}-1s-{date}.zip", timeout=240)
    if code != 200:
        return None
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        k = pd.read_csv(io.BytesIO(z.read([n for n in z.namelist() if n.endswith(".csv")][0])), header=None, names=KCOLS)
    if not str(k.iloc[0, 0]).replace(".", "").isdigit():
        k = k.iloc[1:]
    t = pd.to_numeric(k["open_time"]).to_numpy(); t = np.where(t > 1e14, t // 1000, t)
    t0 = pd.Timestamp(date, tz="UTC").value // 10 ** 6; b = ((t - t0) // (1000 * BAR)).astype(int); nb = 86400 // BAR
    ok = (b >= 0) & (b < nb); qv = pd.to_numeric(k["quote_volume"]).to_numpy(); tb = pd.to_numeric(k["taker_buy_quote_volume"]).to_numpy()
    D = pd.DataFrame({"b": b[ok], "c": pd.to_numeric(k["close"]).to_numpy()[ok], "f": (2 * tb - qv)[ok]}); g = D.groupby("b"); idx = np.arange(nb)
    return g["c"].last().reindex(idx).ffill().bfill().to_numpy(), g["f"].sum().reindex(idx).fillna(0.0).to_numpy() / 1e6
def q_index(X, F):
    Y = np.column_stack([X[1:], F[1:]]); Z = np.column_stack([np.ones(len(Y)), X[:-1], F[:-1]])
    ok = np.all(np.isfinite(Y), 1) & np.all(np.isfinite(Z), 1)
    B = np.linalg.lstsq(Z[ok], Y[ok], rcond=None)[0]; J = B[1:3].T; E = Y[ok] - Z[ok] @ B
    return J[0, 1] * J[1, 0] / (1 - J[1, 1]), E
def unit_co(sym, date):
    r = bars10_spot(sym, date)
    if r is None:
        return None
    c, f = r; p = np.log(c); al = 2 / (EMA_W / BAR + 1); x = p - pd.Series(p).ewm(alpha=al, adjust=False).mean().to_numpy()
    q_ols, E = q_index(x, f)
    rho = float(np.corrcoef(E[1:, 1], E[:-1, 1])[0, 1])                 # AR(1) of flow-equation residuals
    xs, fs = x[1:] - rho * x[:-1], f[1:] - rho * f[:-1]
    q_co, _ = q_index(xs, fs)
    return {"symbol": sym, "date": date, "Q_ols": float(q_ols), "Q_co": float(q_co), "rho_flow_resid": rho}
with ThreadPoolExecutor(6) as ex:
    CO = pd.DataFrame([u for u in ex.map(lambda sd: unit_co(*sd), [(s, d) for s in PAIRS for d in CO_D]) if u])
dm = CO.groupby("date")[["Q_ols", "Q_co", "rho_flow_resid"]].mean()
def mci(v):
    n = len(v); se = v.std(ddof=1) / np.sqrt(n); t = stats.t.ppf(0.975, n - 1); return [float(v.mean() - t * se), float(v.mean() + t * se)]
r7 = {"units": len(CO), "dates": len(dm), "Q_ols": {"mean": float(dm["Q_ols"].mean()), "ci": mci(dm["Q_ols"])},
      "Q_co": {"mean": float(dm["Q_co"].mean()), "ci": mci(dm["Q_co"])}, "rho_flow_resid_median": float(CO["rho_flow_resid"].median()),
      "by_pair": CO.groupby("symbol")[["Q_ols", "Q_co", "rho_flow_resid"]].mean().to_dict("index")}
print(json.dumps({k: v for k, v in r7.items() if k != "by_pair"}, indent=1))
record("R7_cochrane_orcutt_spot", None, "real-data", r7, note="post hoc; Paper 1; 2024 sub-panel of Part 8")
zp = os.path.join(BASE, "p13_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in glob.glob(os.path.join(RESULTS, "R[1-8]_*.json")):
        z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)